# **❓ Missing Values Handling**

In [3]:
import numpy as np
import pandas as pd

---
---
# **🚮 Removal**

---
### └─ **Remove rows with missing values**

In [4]:
def remove_rows_with_placeholder(
    df_train,
    df_test,
    columns,
    placeholder='unknown'
):
    print(f"   └── Removing rows with value '{placeholder}' in columns: {columns}...")

    if df_train is None or df_test is None:
        raise ValueError("Both df_train and df_test must be provided.")

    total_rows_before = len(df_train) + len(df_test)
    total_rows_removed = 0

    def filter_df(df, name):
        df_filtered = df.copy()
        for col in columns:
            if col in df_filtered.columns:
                df_filtered = df_filtered[df_filtered[col] != placeholder]
        removed = len(df) - len(df_filtered)
        print(f"       └── {name}: Removed {removed} rows")
        return df_filtered, removed

    df_train_cleaned, removed_train = filter_df(df_train, "Train")
    df_test_cleaned, removed_test = filter_df(df_test, "Test")

    total_rows_removed = removed_train + removed_test
    total_rows_after = total_rows_before - total_rows_removed
    percentage_remaining = (total_rows_after / total_rows_before) * 100 if total_rows_before > 0 else 0

    print(f"       └── Total rows removed: {total_rows_removed} "
          f"({(total_rows_removed / total_rows_before) * 100:.4f}% of the dataset)")
    print(f"       └── {percentage_remaining:.4f}% of the dataset remains after removal")

    return df_train_cleaned, df_test_cleaned

In [5]:
df_sample_train = pd.DataFrame({
    'name': ['Alice', 'Bob', 'unknown', 'David'],
    'city': ['New York', 'unknown', 'Los Angeles', 'Chicago'],
    'age': [25, 30, 35, 40]
})
df_sample_test = pd.DataFrame({
    'name': ['Eve', 'Frank', 'Grace', 'unknown'],
    'city': ['Houston', 'Phoenix', 'Los Angeles', 'Seattle'],
    'age': [28, 33, 29, 41]
})

df_sample_train, df_sample_test = remove_rows_with_placeholder(
    df_sample_train, 
    df_sample_test,
    columns=['name', 'city'],
    placeholder='unknown'
)
print(f"\nCleaned Train DataFrame:")
df_sample_train
print(f"\nCleaned Test DataFrame:")
df_sample_test

   └── Removing rows with value 'unknown' in columns: ['name', 'city']...
       └── Train: Removed 2 rows
       └── Test: Removed 1 rows
       └── Total rows removed: 3 (37.5000% of the dataset)
       └── 62.5000% of the dataset remains after removal

Cleaned Train DataFrame:

Cleaned Test DataFrame:


,name,city,age
0,Eve,Houston,28
1,Frank,Phoenix,33
2,Grace,Los Angeles,29


---
---
# **🔢 Imputation**

---
### └─ **Impute with logic from other columns**

In [1]:
import pandas as pd

def fill_missing_names(df, id_col, target_col):
    """
    Fill missing values in a target column using mappings from an ID column.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe containing the data.
    id_col : str
        The column containing unique identifiers (e.g., town_id, flatm_id).
    target_col : str
        The column with missing values to fill (e.g., town_name, flatm_name).

    Returns
    -------
    pd.DataFrame
        A copy of the dataframe with missing values filled in the target column.
    """
    df = df.copy()

    # Build mapping from id_col → target_col using rows where target_col is not null
    mapping = df.dropna(subset=[target_col]).drop_duplicates(subset=[id_col]) \
                 .set_index(id_col)[target_col].to_dict()

    # Fill missing values using the mapping
    df[target_col] = df.apply(
        lambda row: mapping.get(row[id_col], row[target_col]),
        axis=1
    )

    return df

In [3]:
# Example dataframe
data = {
    "town_id": [26, 26, 2, 2, 18, 18],
    "town_name": ["Yishun", None, "Bedok", None, "Punggol", None]
}
df = pd.DataFrame(data)

# Fill missing names
df_filled = fill_missing_names(df, "town_id", "town_name")
print(df_filled)

   town_id town_name
0       26    Yishun
1       26    Yishun
2        2     Bedok
3        2     Bedok
4       18   Punggol
5       18   Punggol


---
### └─ **Impute with Aggregate**

In [6]:
def impute_marked_missing_values(
    df_train,
    df_test,
    columns,
    placeholder='unknown',      # You can change this to '999' or 'N/A' and run again
    strategy='mean',            # Try 'mean', 'median', 'mode', or 'constant'
    constant_fill_value=None    # Only needed if strategy is 'constant'
):
    print(f"   └── Replacing placeholder '{placeholder}' using strategy '{strategy}' for columns: {columns}")

    df_train_imputed = df_train.copy()
    df_test_imputed = df_test.copy()

    for col in columns:
        if col not in df_train.columns:
            print(f"       └── Column '{col}' not found in train set. Skipping...")
            continue

        print(f"       └── Imputing column: '{col}'")

        # Replace placeholder temporarily with NaN to allow imputation
        df_train_temp = df_train_imputed[col].replace(placeholder, pd.NA)
        df_test_temp = df_test_imputed[col].replace(placeholder, pd.NA)

        # Determine fill value from train only
        if strategy == 'mean':
            fill_value = pd.to_numeric(df_train_temp, errors='coerce').mean()
        elif strategy == 'median':
            fill_value = pd.to_numeric(df_train_temp, errors='coerce').median()
        elif strategy == 'mode':
            mode_series = df_train_temp.mode()
            fill_value = mode_series.iloc[0] if not mode_series.empty else placeholder
        elif strategy == 'constant':
            fill_value = constant_fill_value
        else:
            raise ValueError(f"Unsupported strategy '{strategy}'")

        # Impute placeholder values
        df_train_imputed[col] = df_train_imputed[col].replace(placeholder, fill_value)
        df_test_imputed[col] = df_test_imputed[col].replace(placeholder, fill_value)

        print(f"           └── Replaced placeholder with: {fill_value}")

    return df_train_imputed, df_test_imputed

In [7]:
df_sample_train = pd.DataFrame({
    'age': ['25', '30', 'unknown', '45', '999'],
    'income': ['50000', '60000', '70000', 'N/A', '55000'],
    'gender': ['male', 'female', 'unknown', 'female', 'male']
})

df_sample_test = pd.DataFrame({
    'age': ['35', 'unknown', '40', '999', '50'],
    'income': ['65000', 'N/A', '62000', '71000', 'unknown'],
    'gender': ['unknown', 'female', 'male', 'unknown', 'female']
})

df_sample_train, df_sample_test = impute_marked_missing_values(
    df_sample_train,
    df_sample_test,
    columns=['age', 'income', 'gender'],
    placeholder='unknown',  # You can change this to '999' or 'N/A' and run again
    strategy='mean',        # Try 'mean', 'median', 'mode', or 'constant'
    # constant_fill_value=0   # Only needed if strategy is 'constant'
)
print(f"\nCleaned Train DataFrame:")
df_sample_train
print(f"\nCleaned Test DataFrame:")
df_sample_test

   └── Replacing placeholder 'unknown' using strategy 'mean' for columns: ['age', 'income', 'gender']
       └── Imputing column: 'age'
           └── Replaced placeholder with: 274.75
       └── Imputing column: 'income'
           └── Replaced placeholder with: 58750.0
       └── Imputing column: 'gender'
           └── Replaced placeholder with: nan

Cleaned Train DataFrame:

Cleaned Test DataFrame:


,age,income,gender
0,35,65000,NaN
1,274.75,N/A,female
2,40,62000,male
3,999,71000,NaN
4,50,58750.0,female


---
### └─ **Impute with KNN**

- A generic imputation technique that can work on numerical data (and sometimes encoded categorical data).

- Imputes missing values based on the values of the nearest neighbors (using Euclidean or other distance metrics).

- Good for numerical data, but less straightforward or accurate for categorical data unless categories are encoded carefully.

In [8]:
from sklearn.impute import KNNImputer
import pandas as pd
import numpy as np

def impute_with_knn(
    df_train,
    df_test,
    columns_for_knn,          # Numeric columns used for computing distance (all must be numeric)
    column_to_impute,         # Single column name to impute
    placeholder='unknown',    # Placeholder value representing missing in the target column
    n_neighbors=5
):
    print(f"   └── KNN Imputation (n={n_neighbors}) for target column: {column_to_impute}")
    print(f"       └── Using these numeric features to compute similarity: {columns_for_knn}")

    df_train_copy = df_train.copy()
    df_test_copy = df_test.copy()

    # Replace placeholder only in the column to impute
    df_train_copy[column_to_impute] = pd.to_numeric(
        df_train_copy[column_to_impute].replace(placeholder, np.nan), errors='coerce'
    )
    df_test_copy[column_to_impute] = pd.to_numeric(
        df_test_copy[column_to_impute].replace(placeholder, np.nan), errors='coerce'
    )

    # Ensure all columns_for_knn are numeric
    train_knn_input = df_train_copy[columns_for_knn].apply(pd.to_numeric, errors='coerce')
    test_knn_input = df_test_copy[columns_for_knn].apply(pd.to_numeric, errors='coerce')

    # Fit KNN imputer
    imputer = KNNImputer(n_neighbors=n_neighbors)
    train_imputed_array = imputer.fit_transform(train_knn_input)
    test_imputed_array = imputer.transform(test_knn_input)

    # Get the dtype of the original column
    original_dtype = df_train[column_to_impute].dtype

    # Extract imputed values into DataFrames
    train_imputed_df = pd.DataFrame(train_imputed_array, columns=columns_for_knn, index=df_train_copy.index)
    test_imputed_df = pd.DataFrame(test_imputed_array, columns=columns_for_knn, index=df_test_copy.index)

    # Replace only the missing values in target column with imputed values, cast safely
    df_train_copy[column_to_impute] = np.where(
        df_train_copy[column_to_impute].isna(),
        train_imputed_df[column_to_impute],
        df_train_copy[column_to_impute]
    ).astype(original_dtype)

    df_test_copy[column_to_impute] = np.where(
        df_test_copy[column_to_impute].isna(),
        test_imputed_df[column_to_impute],
        df_test_copy[column_to_impute]
    ).astype(original_dtype)

    return df_train_copy, df_test_copy

In [9]:
# Sample training DataFrame
df_sample_train = pd.DataFrame({
    'age': [25, 30, -999, 45, 60],
    'income': [50000, 60000, 70000, 55000, 55000],
    'score': [80, 85, 90, 45, 75],
    'gender': ['male', 'female', 'female', 'unknown', 'male']
})

# Sample testing DataFrame
df_sample_test = pd.DataFrame({
    'age': [35, -999, 40, 20, 50],
    'income': [40000, 50000, 62000, 71000, 80000],
    'score': [85, 74, 88, 90, 82],
    'gender': ['female', 'female', 'male', 'unknown', 'female']
})

df_sample_train, df_sample_test = impute_with_knn(
    df_sample_train,
    df_sample_test,
    columns_for_knn=['age', 'income', 'score'],  # Must be numeric or convertible
    column_to_impute='age',
    placeholder=-999,                     # You can also run it for '999' and 'N/A'
    n_neighbors=3
)
print(f"\nCleaned Train DataFrame:")
df_sample_train
print(f"\nCleaned Test DataFrame:")
df_sample_test

   └── KNN Imputation (n=3) for target column: age
       └── Using these numeric features to compute similarity: ['age', 'income', 'score']

Cleaned Train DataFrame:

Cleaned Test DataFrame:


,age,income,score,gender
0,35,40000,85,female
1,43,50000,74,female
2,40,62000,88,male
3,20,71000,90,unknown
4,50,80000,82,female


---
### └─ **Impute with K-Modes**

- Designed specifically for categorical data clustering.

- Uses a simple matching dissimilarity measure (count of mismatches) for categorical variables.

- Best when your dataset (or clustering variables) are purely categorical.

- Can be used for imputation by assigning cluster modes to missing values.

In [10]:
import pandas as pd
import numpy as np
from kmodes.kmodes import KModes

def impute_with_kmodes(
    df_train,
    df_test,
    columns_for_clustering,    # All categorical columns used for clustering (distance)
    column_to_impute,          # Single categorical column to actually impute
    placeholder='unknown',     # Placeholder string representing missing value
    n_clusters=5,
    random_state=42
):
    print(f"   └── KModes Imputation (k={n_clusters}) for target column: {column_to_impute}")
    print(f"       └── Using these categorical features to cluster similarity: {columns_for_clustering}")

    df_train_copy = df_train.copy()
    df_test_copy = df_test.copy()

    # Replace placeholder with np.nan for clustering and imputation in clustering columns
    for col in columns_for_clustering:
        df_train_copy[col] = df_train_copy[col].replace(placeholder, np.nan)
        df_test_copy[col] = df_test_copy[col].replace(placeholder, np.nan)

    # Fit KModes on training data (only clustering columns)
    train_cluster_data = df_train_copy[columns_for_clustering].fillna('missing_placeholder').astype(str)
    km = KModes(n_clusters=n_clusters, init='Huang', n_init=5, random_state=random_state)
    cluster_labels = km.fit_predict(train_cluster_data)

    # Get cluster modes (most frequent category) for each cluster & each column
    cluster_modes = pd.DataFrame(km.cluster_centroids_, columns=columns_for_clustering)

    # Helper function to impute only the target column; others keep original values (including placeholder)
    def impute_row(row, cluster_label, original_row):
        if (pd.isna(row[column_to_impute]) or original_row[column_to_impute] == placeholder):
            row[column_to_impute] = cluster_modes.loc[cluster_label, column_to_impute]
        # Keep all other columns unchanged
        for col in columns_for_clustering:
            if col != column_to_impute:
                row[col] = original_row[col]
        return row

    # Impute train set
    df_train_copy['__cluster'] = cluster_labels
    df_train_copy = df_train_copy.apply(
        lambda row: impute_row(row, row['__cluster'], df_train.loc[row.name]),
        axis=1
    )
    df_train_copy.drop(columns='__cluster', inplace=True)

    # Assign clusters to test set by finding closest cluster mode (Hamming distance)
    def assign_cluster(row):
        row_vals = row[columns_for_clustering].fillna('missing_placeholder').astype(str).values
        distances = (cluster_modes.values != row_vals).sum(axis=1)
        return np.argmin(distances)

    df_test_copy['__cluster'] = df_test_copy.apply(assign_cluster, axis=1)
    df_test_copy = df_test_copy.apply(
        lambda row: impute_row(row, row['__cluster'], df_test.loc[row.name]),
        axis=1
    )
    df_test_copy.drop(columns='__cluster', inplace=True)

    return df_train_copy, df_test_copy

In [11]:
# Sample training data
df_sample_train = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'unknown', 'blue', 'green', 'red', 'unknown', 'green'],
    'shape': ['circle', 'square', 'triangle', 'circle', 'circle', 'triangle', 'circle', 'square', 'circle'],
    'size': ['small', 'medium', 'large', 'small', 'medium', 'large', 'small', 'medium', 'large']
})

# Sample test data
df_sample_test = pd.DataFrame({
    'color': ['blue', 'unknown', 'red', 'green', 'unknown'],
    'shape': ['triangle', 'triangle', 'square', 'square', 'circle'],
    'size': ['medium', 'large', 'small', 'large', 'medium']
})

# Call the KModes imputation function
df_sample_train, df_sample_test = impute_with_kmodes(
    df_sample_train,
    df_sample_test,
    columns_for_clustering=['color', 'shape', 'size'],
    column_to_impute='color',
    placeholder='unknown',
    n_clusters=3,
    random_state=42
)

print("\nImputed Train DataFrame:")
df_sample_train
print("\nImputed Test DataFrame:")
df_sample_test

   └── KModes Imputation (k=3) for target column: color
       └── Using these categorical features to cluster similarity: ['color', 'shape', 'size']

Imputed Train DataFrame:

Imputed Test DataFrame:


,color,shape,size
0,blue,triangle,medium
1,green,triangle,large
2,red,square,small
3,green,square,large
4,blue,circle,medium


---
### └─ **Impute with K-Prototypes**

- Extension of K-Modes that can handle mixed data types (both categorical and numerical).

- Combines Euclidean distance for numeric features + matching dissimilarity for categorical ones.

- If your dataset has both numerical and categorical features, this is often the better choice.

- Clustering with K-Prototypes better captures similarities in mixed-type datasets.

In [12]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes

def impute_with_kprototypes(
    df_train,
    df_test,
    columns_for_clustering,       # All columns used for clustering (categorical + numeric)
    column_to_impute,             # The single column to impute
    cat_placeholder='unknown',    # Placeholder for missing categorical values
    num_placeholder=-999,         # Placeholder for missing numeric values
    n_clusters=5,
    random_state=42
):
    print(f"   └── KPrototypes Imputation (k={n_clusters}) for target column: {column_to_impute}")
    print(f"       └── Using these features to compute similarity: {columns_for_clustering}")

    df_train_copy = df_train.copy()
    df_test_copy = df_test.copy()

    # Auto-detect categorical columns
    categorical_columns = [col for col in columns_for_clustering if df_train_copy[col].dtype == 'object' or df_train_copy[col].dtype.name == 'category']
    cat_cols_idx = [columns_for_clustering.index(col) for col in categorical_columns]
    is_categorical = column_to_impute in categorical_columns
    numeric_columns = [col for col in columns_for_clustering if col not in categorical_columns]

    # Replace missing in the column to impute only
    if is_categorical:
        df_train_copy[column_to_impute] = df_train_copy[column_to_impute].replace(cat_placeholder, np.nan)
        df_test_copy[column_to_impute] = df_test_copy[column_to_impute].replace(cat_placeholder, np.nan)
    else:
        df_train_copy[column_to_impute] = df_train_copy[column_to_impute].replace(num_placeholder, np.nan)
        df_test_copy[column_to_impute] = df_test_copy[column_to_impute].replace(num_placeholder, np.nan)

    # Replace all numeric placeholders in other numeric columns
    for col in numeric_columns:
        df_train_copy[col] = df_train_copy[col].replace(num_placeholder, np.nan)
        df_test_copy[col] = df_test_copy[col].replace(num_placeholder, np.nan)

    # Fill missing for clustering
    train_cluster_data = df_train_copy[columns_for_clustering].copy()

    for col in categorical_columns:
        train_cluster_data[col] = train_cluster_data[col].fillna('missing_placeholder').astype(str)

    for col in numeric_columns:
        train_cluster_data[col] = pd.to_numeric(train_cluster_data[col], errors='coerce')
        train_cluster_data[col] = train_cluster_data[col].fillna(train_cluster_data[col].mean())

    # KPrototypes model
    train_np = train_cluster_data.to_numpy()
    cat_cols_idx = [columns_for_clustering.index(col) for col in categorical_columns]

    kproto = KPrototypes(n_clusters=n_clusters, init='Huang', n_init=5, verbose=0, random_state=random_state)
    cluster_labels = kproto.fit_predict(train_np, categorical=cat_cols_idx)
    cluster_centroids = kproto.cluster_centroids_
    cluster_modes_df = pd.DataFrame(cluster_centroids, columns=columns_for_clustering)

    # Row-level imputer
    def impute_row(row, cluster_label, original_val):
        centroid_val = cluster_modes_df.loc[cluster_label, column_to_impute]
        if is_categorical:
            if pd.isna(row[column_to_impute]) or original_val == cat_placeholder:
                row[column_to_impute] = centroid_val
        else:
            try:
                missing = pd.isna(row[column_to_impute]) or pd.isna(original_val) or original_val == num_placeholder
            except:
                missing = True
            if missing:
                row[column_to_impute] = float(centroid_val)
        return row

    # Impute training set
    df_train_copy['__cluster'] = cluster_labels
    df_train_copy = df_train_copy.apply(
        lambda row: impute_row(row, row['__cluster'], df_train.loc[row.name, column_to_impute]), axis=1
    )
    df_train_copy.drop(columns='__cluster', inplace=True)

    # Assign clusters to test set
    def assign_cluster(row):
        row_vals = []
        for col in columns_for_clustering:
            val = row[col]
            if col in categorical_columns:
                val = str(val) if pd.notna(val) else 'missing_placeholder'
            else:
                try:
                    val = float(val)
                except:
                    val = np.nan
            row_vals.append(val)

        dists = []
        for i, centroid in enumerate(cluster_centroids):
            dist = 0
            for idx, col in enumerate(columns_for_clustering):
                if idx in cat_cols_idx:
                    dist += 1 if centroid[idx] != row_vals[idx] else 0
                else:
                    try:
                        c_val = float(centroid[idx])
                    except:
                        c_val = np.nan
                    if np.isnan(row_vals[idx]) or np.isnan(c_val):
                        dist += 1e6
                    else:
                        dist += (c_val - row_vals[idx]) ** 2
            dists.append(dist)
        return np.argmin(dists)

    df_test_copy['__cluster'] = df_test_copy.apply(assign_cluster, axis=1)
    df_test_copy = df_test_copy.apply(
        lambda row: impute_row(row, row['__cluster'], df_test.loc[row.name, column_to_impute]), axis=1
    )
    df_test_copy.drop(columns='__cluster', inplace=True)

    return df_train_copy, df_test_copy

In [13]:
df_sample_train = pd.DataFrame({
    'age': [25, 30, 22, -999, 40, 35, 28, 33, -999],
    'income': [50000, 60000, 52000, 58000, 60000, 62000, 58000, 59000, 61000],
    'marital_status': ['single', 'married', 'single', 'married', 'married', 'divorced', 'married', 'single', 'married'],
    'education': ['bachelor', 'master', 'bachelor', 'phd', 'married', 'master', 'bachelor', 'married', 'phd']
})

df_sample_test = pd.DataFrame({
    'age': [34, -999, 29, 38, 41, 36, 32, 30],
    'income': [55000, 57000, 60000, 61000, 60000, 58000, 59000, 70000],
    'marital_status': ['married', 'single', 'married', 'married', 'divorced', 'married', 'single', 'married'],
    'education': ['master', 'bachelor', 'phd', 'married', 'bachelor', 'master', 'phd', 'married']
})

df_sample_train, df_sample_test = impute_with_kprototypes(
    df_sample_train,
    df_sample_test,
    columns_for_clustering=['age', 'income', 'marital_status', 'education'],
    column_to_impute='age',
    cat_placeholder='unknown',         # categorical missing placeholder
    num_placeholder=-999,              # numeric missing placeholder
    n_clusters=3,
    random_state=42
)

print("\nImputed Train DataFrame:")
df_sample_train
print("\nImputed Test DataFrame:")
df_sample_test

   └── KPrototypes Imputation (k=3) for target column: age
       └── Using these features to compute similarity: ['age', 'income', 'marital_status', 'education']

Imputed Train DataFrame:

Imputed Test DataFrame:


,age,income,marital_status,education
0,34.00000,55000,married,master
1,30.47619,57000,single,bachelor
2,29.00000,60000,married,phd
3,38.00000,61000,married,married
4,41.00000,60000,divorced,bachelor
5,36.00000,58000,married,master
6,32.00000,59000,single,phd
7,30.00000,70000,married,married


---
---
---